# KweriMoto: YAMNet baseline

**Frozen YAMNet features + a small three-class classifier** (healthy / engine_knock / misfire).

[Open in Colab](https://colab.research.google.com/github/SUUNARAYMOND/kwerimoto-dataset/blob/main/notebooks/yamnet_baseline.ipynb) · Dataset: [rayyyyy007/kwerimoto-engine-sounds](https://huggingface.co/datasets/rayyyyy007/kwerimoto-engine-sounds)

```
clip (16 kHz) → YAMNet (frozen) → 1,024 numbers per 0.96 s piece → combine per clip → small classifier → label
```

| # | Section |
|---|---|
| 0 | Setup |
| 1 | Load the dataset |
| 2 | Look at the data |
| 3 | Get YAMNet features |
| 4 | Sanity check: what does YAMNet hear? |
| 5 | Train the head |
| 6 | Test (once) |
| 7 | `real_test` |
| 8 | Check the mistakes |
| 9 | Save |

**Rules:** train on `train`, choose settings on `validation`, score `test` once at the end, never train or tune on `real_test`.

## 0. Setup

Fixed settings in one place. `SEED` makes random choices repeat on every run. On Colab, results are saved
to your Google Drive so they survive when the session ends.

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display

DATASET_ID = "rayyyyy007/kwerimoto-engine-sounds"
LABELS = ["healthy", "engine_knock", "misfire"]
SAMPLE_RATE = 16_000
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:  # on Colab: save to Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = Path("/content/drive/MyDrive/kwerimoto/yamnet")
except ImportError:  # anywhere else: save next to the notebook
    SAVE_DIR = Path("outputs/yamnet")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("Saving results to", SAVE_DIR)

## 1. Load the dataset

`snapshot_download` copies the dataset from Hugging Face (about 290 MB, once per session) and returns the folder.
`manifest.csv` has one row per clip: its file, label, split and source. We add `path`, the full path to each clip.

In [ ]:
from huggingface_hub import snapshot_download

root = Path(snapshot_download(DATASET_ID, repo_type="dataset"))
manifest = pd.read_csv(root / "manifest.csv", keep_default_na=False)
manifest["path"] = [str(root / f) for f in manifest["file_name"]]
print(len(manifest), "clips")
manifest.head(3)

## 2. Look at the data

Before training anything, check what we have: how many clips per label and split, and how long they are.

In [ ]:
pd.crosstab([manifest["label"], manifest["source_dataset"]], manifest["split"], margins=True, margins_name="total")[
    ["train", "validation", "test", "real_test", "total"]]

**Clip lengths.** YAMNet looks at 0.96 s pieces, so a clip shorter than 0.96 s gives a single piece that is
partly silence. Most of these short clips are synthetic knock, which is why we check later whether the model's
mistakes depend on length.

In [ ]:
display(manifest.groupby(["label", "source_dataset"])["duration_s"].describe()[["count", "min", "50%", "max"]].round(2))

short = manifest[manifest["duration_s"] < 0.96]
print(f"{len(short)} clips are shorter than one YAMNet piece (0.96 s):", short.groupby("label").size().to_dict())

fig, ax = plt.subplots(figsize=(8, 3))
for label in LABELS:
    ax.hist(manifest.loc[manifest["label"] == label, "duration_s"], bins=40, alpha=0.6, label=label)
ax.set_xlabel("clip length (s)"); ax.set_ylabel("clips"); ax.legend(); plt.show()

**Listen.** One random training clip per label.

In [ ]:
for label in LABELS:
    row = manifest[(manifest["split"] == "train") & (manifest["label"] == label)].sample(1, random_state=SEED).iloc[0]
    print(f"{label}: {row['file_name']} ({row['duration_s']} s, from {row['source_dataset']})")
    display(Audio(row["path"]))